# **Report 1: Analysing Matrix Monitoring Methods In Accelerating Frequent Pattern Recognition Algorithms**

### **Professor:** Dr. Ghatee

### **Head TA:** Behnam Yousefimehr

### **Author:** Hassan Hajizadeh

## **Step 1: Downloading, Loading And Preprocessing The Data From [UCI Website](https://archive.ics.uci.edu/dataset/352/online+retail)**

We already downloaded the Dataset and for properly loading the file we will install pandas and openpyxl (for loading xlsx) libraries.

We also install numpy library for calculations and scikit-learn for using its algorithm methods.

And we will install matplotlib and seaborn libraries for plotting our data and results.

In [1]:
!pip install pandas
!pip install openpyxl
!pip install numpy
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn

### **Loading Data**

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('Online Retail.xlsx')

### **Cleaning Data**

Based on what we see below in dataset info, indicates that there are some empty values in the description and the CustomerID Features.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


#### **Editing Or Dropping Bad Samples**

Based on the project description, we will just need the **InvoiceNo** and **StockCode** and **InvoiceDate** columns for the rest of the project.

So dropping almost **140,000** rows of data, just because of **ONE** column, the **CustomerID** column, Would Be unnecessary.

And also Based on the **StockCode** Description in UCI Dataset page in the website, mentions that each product is **"a 5-digit integral number uniquely assigned to each distinct product"** so we will drop the rows that their **StockCode** doesn't Start with a **5-digit integral number**.

In the mean time, we will remove the letter at the end of some **StockCodes** because it's just the color and type letter and we don't want to make another category for each color of a product too.

And we will also drop every row that their **InvoiceNo** is not just **a 6-digits number**. beacuse of description of **InvoiceNo** values that mentions it should be **"a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation."**

In [4]:
df['StockCode'] = df['StockCode'].astype(str).str.strip()
mask = df['StockCode'].str.match(r'^\d{5}', na=False)
df = df[mask].copy()
df['StockCode'] = df['StockCode'].str.extract(r'^(\d{5})')

df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()
mask = ~df['InvoiceNo'].str.fullmatch(r'\d{6}', na=False)
df = df.drop(df[mask].index)

#### **Checking And Removing The Negative Data**

In [5]:
(df[["Quantity","UnitPrice","CustomerID"]] < 0).any().any()

np.True_

In [6]:
df[df[["Quantity","UnitPrice","CustomerID"]] < 0].stack()

2406    Quantity     -10.0
4347    Quantity     -38.0
7188    Quantity     -20.0
7189    Quantity     -20.0
7190    Quantity      -6.0
                     ...  
535333  Quantity     -26.0
535335  Quantity   -1050.0
535336  Quantity     -30.0
536908  Quantity    -338.0
538919  Quantity    -235.0
Length: 1324, dtype: object

In [7]:
df = df[(df['Quantity'] >= 0) & (df['UnitPrice'] >= 0)]

In [8]:
(df[["Quantity","UnitPrice","CustomerID"]] < 0).any().any()

np.False_

**PS: By This Simple Tricks We Saved 130,962 Rows Of Data From Getting Dropped**

## **Step 2: Creating Transaction-Item Binary Matrix**

Based on matrix below we will use InvoiceNo and StockCode categories to create our binary matrix.

In [9]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [10]:
binary_matrix = pd.crosstab(df['InvoiceNo'], df['StockCode'])
binary_matrix = (binary_matrix > 0).astype(int)
binary_matrix


StockCode,10002,10080,10120,10123,10124,10125,10133,10135,11001,15030,...,90202,90204,90205,90206,90208,90209,90210,90211,90212,90214
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581585,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Step 3: Streaming Simulation For Binary Matrix**

For streaming simulation first we sort the binary matrix based on InvoiceDate.

In [11]:
first_time_for_each_transaction = df.groupby('InvoiceNo')['InvoiceDate'].min()
sorted_binary_matrix = binary_matrix.loc[first_time_for_each_transaction.sort_values().index]


Then we divide this sorted binary matrix to 14 batches.

**PS: I just choosed 14, to our batche sizes be equal togather.**

In [12]:
number_of_batches = 14
batch_size = int(np.ceil(len(sorted_binary_matrix) / number_of_batches))
batches = [
    sorted_binary_matrix.iloc[i*batch_size : (i+1)*batch_size]
    for i in range(number_of_batches)
]


## **Step 4: Matrix Monitoring Using Different Algorithms**

In this section we will use matrix skeching methods for increasing speed and decreasing memory usage as well as preserving main statistical information.

### **First Matrix Skeching Method: Gaussian Random Projection**

In [13]:
def generate_random_matrix(number_of_samples , original_dimension , new_dimension=None , eps=None):
    if(new_dimension == None and eps==None):
        raise ValueError("Both of new_dimension and eps couldn't be None togather")
    if (eps != None):
        new_dimension = int(4 * np.log(number_of_samples) / (eps**2 / 2 - eps**3 / 3))
    random_matrix = np.random.normal(0,1/np.sqrt(new_dimension),size=(original_dimension,new_dimension))
    return random_matrix
    
def gaussian_random_projection(matrix , random_matrix):
    return np.dot(matrix,random_matrix)


### **Second Matrix Skeching Method: Incremental PCA**

In [14]:
import numpy as np

class Incremental_PCA:
    def __init__(self, new_dim):
        self.mean_ = None
        self.new_dim = new_dim
        self.samples_seen = 0
        self.cov_ = None

    def partial_fit(self, X):
        X = np.asarray(X)
        n_new = X.shape[0]
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
            self.cov_ = np.cov(X, rowvar=False) * (n_new - 1)
            self.samples_seen = n_new
        else:
            old_mean = self.mean_
            new_mean = np.mean(X, axis=0)
            total_n = self.samples_seen + n_new
            updated_mean = (self.samples_seen * old_mean + n_new * new_mean) / total_n
            X_centered = X - new_mean
            new_cov = X_centered.T @ X_centered
            mean_diff = (old_mean - new_mean).reshape(-1, 1)
            mean_correction = (self.samples_seen * n_new) / total_n * (mean_diff @ mean_diff.T)
            updated_cov = self.cov_ + new_cov + mean_correction
            self.mean_ = updated_mean
            self.cov_ = updated_cov
            self.samples_seen = total_n

    def components_(self):
        if self.cov_ is None:
            return None
        eigvals, eigvecs = np.linalg.eigh(self.cov_ / (self.samples_seen - 1))
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        return eigvecs[:, order[:self.new_dim]].T , eigvals

    def transform(self, X):
        X = np.asarray(X)
        components , eigvals = self.components_()
        if self.mean_ is not None:
            X_c = X - self.mean_
        else:
            X_c = X
        return X_c @ components.T , eigvals

### **Third Matrix Skeching Method: Frequent Directions**

In [15]:
import numpy as np

class Frequent_Directions:
    def __init__(self, n_columns: int, ell: int):
        self.ell = ell
        self.B = np.zeros((0, n_columns))

    def compress(self):
        U, s, Vt = np.linalg.svd(self.B, full_matrices=False)
        delta = s[self.ell - 1] ** 2
        s_compress = np.sqrt(np.maximum(s**2 - delta, 0))
        self.B = np.diag(s_compress[:self.ell]) @ Vt[:self.ell, :]

    def fit(self, batch):
        A_batch = batch if isinstance(batch, np.ndarray) else np.asarray(batch)
        self.B = np.vstack([self.B, A_batch])
        if self.B.shape[0] > 2 * self.ell:
            self.compress()

    def transform(self):
        if self.B.shape[0] > self.ell:
            self.compress()
        return self.B


### **Feeding Data Batches To Gaussian Random Projection Algorithm**

In [16]:
rows , columns = batches[0].shape
R = generate_random_matrix(number_of_samples=rows , original_dimension=columns , new_dimension=20)
grp_result = []
for batch in batches:
    grp_result.append(gaussian_random_projection(batch,R))
grp_result = np.array(grp_result)
grp_result.shape

(14, 1467, 20)

### **Feeding Data Batches To Incremental PCA Algorithm**

In [17]:
ipca_new_dim = 1000
ipca = Incremental_PCA(new_dim=ipca_new_dim)
ipca_result = []
ipca_eigvals = []
for i in range(len(batches)):
    ipca.partial_fit(batches[i])
    result , eigvals = ipca.transform(batches[i])
    ipca_result.append(result)
    ipca_eigvals.append(eigvals)
    
ipca_result = np.array(ipca_result)
ipca_eigvals = np.array(ipca_eigvals)
ipca_result.shape

(14, 1467, 1000)

### **Feeding Data Batches To Frequent Directions Algorithm**

In [18]:
n_columns = batches[0].shape[1]
fd_result = []
for batch in batches:
    fd = Frequent_Directions(n_columns = n_columns, ell=300)
    fd.fit(batch)
    fd_result.append(fd.transform())
fd_result = np.array(fd_result)
fd_result.shape

(14, 300, 3296)

PS: As we see GRP and IPCA are Features Reduction algorithms on the otherhand, Frequent Directions is a Sample Reduction algorithm.

## **Step 5: Calculation Of Analytical Indicators**

### **Calculating Frobenius Norm**

#### **Calculating Frobenius Norm For original Data**

In [19]:
for i in range(len(batches)):
    print("Frobenius Norm of "+str(i+1)+"th batch of original data: ",np.linalg.norm(batches[i],'fro'))

Frobenius Norm of 1th batch of original data:  188.60540819393276
Frobenius Norm of 2th batch of original data:  200.68383093812017
Frobenius Norm of 3th batch of original data:  184.70517047446182
Frobenius Norm of 4th batch of original data:  178.5609139761555
Frobenius Norm of 5th batch of original data:  177.9213309302738
Frobenius Norm of 6th batch of original data:  173.51080657987848
Frobenius Norm of 7th batch of original data:  182.67183690979843
Frobenius Norm of 8th batch of original data:  187.3125729896421
Frobenius Norm of 9th batch of original data:  186.38401218988716
Frobenius Norm of 10th batch of original data:  195.53772014626742
Frobenius Norm of 11th batch of original data:  197.46392075515973
Frobenius Norm of 12th batch of original data:  196.5400722499104
Frobenius Norm of 13th batch of original data:  201.8192260415246
Frobenius Norm of 14th batch of original data:  207.81482141560548


#### **Calculating Frobenius Norm For GRP Results**

In [20]:
for i in range(len(grp_result)):
    print("Frobenius Norm of "+str(i+1)+"th batch of GRP results: ",np.linalg.norm(grp_result[i],'fro'))

Frobenius Norm of 1th batch of GRP results:  192.5136886117465
Frobenius Norm of 2th batch of GRP results:  199.89497265607892
Frobenius Norm of 3th batch of GRP results:  186.31640435335947
Frobenius Norm of 4th batch of GRP results:  179.56188209504987
Frobenius Norm of 5th batch of GRP results:  180.62509201618343
Frobenius Norm of 6th batch of GRP results:  175.10379655248573
Frobenius Norm of 7th batch of GRP results:  182.3834790717687
Frobenius Norm of 8th batch of GRP results:  187.15845621396662
Frobenius Norm of 9th batch of GRP results:  187.76161379898195
Frobenius Norm of 10th batch of GRP results:  195.74248157934488
Frobenius Norm of 11th batch of GRP results:  201.0586365611243
Frobenius Norm of 12th batch of GRP results:  201.89167657170614
Frobenius Norm of 13th batch of GRP results:  207.12157571597314
Frobenius Norm of 14th batch of GRP results:  216.04892729204263


#### **Calculating Frobenius Norm For IPCA Results**

In [21]:
for i in range(len(ipca_result)):
    print("Frobenius Norm of "+str(i+1)+"th batch of IPCA results: ",np.linalg.norm(ipca_result[i],'fro'))

Frobenius Norm of 1th batch of IPCA results:  185.67478202540656
Frobenius Norm of 2th batch of IPCA results:  195.1645613504764
Frobenius Norm of 3th batch of IPCA results:  177.08133228017334
Frobenius Norm of 4th batch of IPCA results:  169.61773948958916
Frobenius Norm of 5th batch of IPCA results:  167.70689460559586
Frobenius Norm of 6th batch of IPCA results:  162.3677817541636
Frobenius Norm of 7th batch of IPCA results:  170.32084360977115
Frobenius Norm of 8th batch of IPCA results:  173.81217647117904
Frobenius Norm of 9th batch of IPCA results:  171.6237815530511
Frobenius Norm of 10th batch of IPCA results:  177.7296607414394
Frobenius Norm of 11th batch of IPCA results:  178.94229977360715
Frobenius Norm of 12th batch of IPCA results:  177.96378025601842
Frobenius Norm of 13th batch of IPCA results:  184.29000318737647
Frobenius Norm of 14th batch of IPCA results:  190.45339842474826


#### **Calculating Frobenius Norm For Frequent Directions Results**

In [22]:
for i in range(len(fd_result)):
    print("Frobenius Norm of "+str(i+1)+"th batch of FD results: ",np.linalg.norm(fd_result[i],'fro'))

Frobenius Norm of 1th batch of FD results:  151.08930061623923
Frobenius Norm of 2th batch of FD results:  156.84676634562936
Frobenius Norm of 3th batch of FD results:  138.49303953957926
Frobenius Norm of 4th batch of FD results:  133.87953386051527
Frobenius Norm of 5th batch of FD results:  134.989836508666
Frobenius Norm of 6th batch of FD results:  131.1306153081395
Frobenius Norm of 7th batch of FD results:  140.72482868911334
Frobenius Norm of 8th batch of FD results:  143.0561575314455
Frobenius Norm of 9th batch of FD results:  141.873622643369
Frobenius Norm of 10th batch of FD results:  145.60053894187212
Frobenius Norm of 11th batch of FD results:  147.58485817708927
Frobenius Norm of 12th batch of FD results:  147.77250063653776
Frobenius Norm of 13th batch of FD results:  155.6347606381093
Frobenius Norm of 14th batch of FD results:  165.54541903561895


### **Calculating Explained Variance Ratio**

I only calculate this for IPCA algorithm cause it's for PCA Like algorithms ONLY.

#### **Calculating EVR For IPCA**

In [23]:
for i in range(len(ipca_eigvals)):
    print("EVR sum for batch"+str(i)+"th is: ",np.sum(ipca_eigvals[:ipca_new_dim]/np.sum(ipca_eigvals)))

EVR sum for batch0th is:  1.0
EVR sum for batch1th is:  1.0
EVR sum for batch2th is:  1.0
EVR sum for batch3th is:  1.0
EVR sum for batch4th is:  1.0
EVR sum for batch5th is:  1.0
EVR sum for batch6th is:  1.0
EVR sum for batch7th is:  1.0
EVR sum for batch8th is:  1.0
EVR sum for batch9th is:  1.0
EVR sum for batch10th is:  1.0
EVR sum for batch11th is:  1.0
EVR sum for batch12th is:  1.0
EVR sum for batch13th is:  1.0


This means almost 100% of the variance is explained by all components combined.

### **Calculating Reconstruction Error**

#### **GRP Reconstruction Error**

In [24]:
for i in range(len(batches)):
    reconstruct_grp = grp_result[i] @ np.linalg.pinv(R)
    grp_recon_err = np.linalg.norm(batches[i]-reconstruct_grp,'fro')/np.linalg.norm(batches[i], 'fro')**2
    print("Reconstruction Error for the batch "+str(i+1)+"th in GRP method is: ",grp_recon_err)

Reconstruction Error for the batch 1th in GRP method is:  0.005285006118981136
Reconstruction Error for the batch 2th in GRP method is:  0.004967750913656536
Reconstruction Error for the batch 3th in GRP method is:  0.005397130862644025
Reconstruction Error for the batch 4th in GRP method is:  0.005582927775310859
Reconstruction Error for the batch 5th in GRP method is:  0.0056026951036389095
Reconstruction Error for the batch 6th in GRP method is:  0.005745328021611823
Reconstruction Error for the batch 7th in GRP method is:  0.00545749954218985
Reconstruction Error for the batch 8th in GRP method is:  0.0053222347382011
Reconstruction Error for the batch 9th in GRP method is:  0.0053485163151644035
Reconstruction Error for the batch 10th in GRP method is:  0.005098351545528471
Reconstruction Error for the batch 11th in GRP method is:  0.005048059001489742
Reconstruction Error for the batch 12th in GRP method is:  0.005071445642900917
Reconstruction Error for the batch 13th in GRP met

#### **IPCA Reconstruction Error**

In [25]:
comp,_ = ipca.components_()
final_mean = ipca.mean_
comp = np.linalg.pinv(comp).T
for i in range(len(ipca_result)):
    approximate = ipca_result[i] @ comp + final_mean
    recon_err = np.linalg.norm(batches[i]-approximate ,'fro') / np.linalg.norm(batches[i],'fro')
    print("Reconstruction Error for the batch "+str(i)+"th in IPCA method is:",recon_err)

Reconstruction Error for the batch 0th in IPCA method is: 1.4808979971998413
Reconstruction Error for the batch 1th in IPCA method is: 1.340484116951727
Reconstruction Error for the batch 2th in IPCA method is: 1.357231798291484
Reconstruction Error for the batch 3th in IPCA method is: 1.3531025013212161
Reconstruction Error for the batch 4th in IPCA method is: 1.4064713792039916
Reconstruction Error for the batch 5th in IPCA method is: 1.3547783612944027
Reconstruction Error for the batch 6th in IPCA method is: 1.3377906211834496
Reconstruction Error for the batch 7th in IPCA method is: 1.3782885106993323
Reconstruction Error for the batch 8th in IPCA method is: 1.3774737682607319
Reconstruction Error for the batch 9th in IPCA method is: 1.3208758094643476
Reconstruction Error for the batch 10th in IPCA method is: 1.3568434153024138
Reconstruction Error for the batch 11th in IPCA method is: 1.371480486423411
Reconstruction Error for the batch 12th in IPCA method is: 1.4110933016277758

#### **Frequent Directions Reconstruction Error**